In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly as py 
import plotly.tools as tls 
import plotly.express as px

In [ ]:
marks= pd.read_excel("C:/Data_Science/Pandas/Student Performance Analysis.xlsx",sheet_name='Marks')
attendance= pd.read_excel("C:/Data_Science/Pandas/Student Performance Analysis.xlsx",sheet_name='Attendance')

In [ ]:
marks['Name']= marks['Name'].str.strip().str.title()
attendance['Name']= attendance['Name'].str.strip().str.title()

In [ ]:
#replacing space & null with 0
marks=marks.replace(' ',0)
marks=marks.fillna(0)
#Replace attendance values ('Y' or 'N') with numeric values (1 for 'Y' and 0 for 'N')
attendance.replace('Y',1,inplace=True)
attendance.replace('N',0,inplace=True)

In [ ]:
#casting columns to perform calculation
marks['Mini Test 1']=marks['Mini Test 1'].astype('int64')
marks['Mini Test 2']=marks['Mini Test 2'].astype('int64')
marks['Live Test']=marks['Live Test'].astype('int64')
marks['Assignment']=marks['Assignment'].astype('int64')

In [ ]:
##marks.to_csv('C:/Data_Science/Pandas/test.csv')

In [ ]:
marks.replace(113,15,inplace=True)

In [ ]:
#marks[(marks['Name']=='Sophia Hardy')]

In [ ]:
#merge both sheets
student_performance= pd.merge(marks,attendance,on='Name',how='left')

In [ ]:
#creating column Total Marks
student_performance['Total Marks'] = student_performance[['Mini Test 1','Mini Test 2','Live Test','Assignment']].sum(axis=1) #summing horizontally
student_performance.groupby('Name').sum()

In [ ]:
student_performance['Percentage'] = round((student_performance['Total Marks']/max_marks*100),1)
student_performance['Total Attendance']=student_performance[['Attendance Day 1','Attendance Day 2','Attendance Day 3','Attendance Day 4','Attendance Day 5']].sum(axis=1)
#attedance percentage
student_performance['Attendance Percentage'] = student_performance[['Attendance Day 1','Attendance Day 2','Attendance Day 3','Attendance Day 4','Attendance Day 5']].mean(axis=1) * 100

In [ ]:
#calcuting maximum marks
mini_test1_max = max(student_performance['Mini Test 1'])
mini_test2_max = max(student_performance['Mini Test 2'])
live_test_max = max(student_performance['Live Test'])
assignment_max = max(student_performance['Assignment'])
print(mini_test1_max)
print(mini_test2_max)
print(live_test_max)
print(assignment_max)

In [ ]:
#student_performance['test1_percent'] = 
student_performance['percentage_mini_test1'] = student_performance['Mini Test 1']/mini_test1_max
student_performance['percentage_mini_test2'] = student_performance['Mini Test 2']/mini_test2_max
student_performance['percentage_live_test'] = student_performance['Live Test']/live_test_max
student_performance['Assignment'] = student_performance['Assignment']/assignment_max

In [ ]:
student_performance.head(5)

In [ ]:
attendance_per= student_performance[['Attendance Day 1','Attendance Day 2','Attendance Day 3','Attendance Day 4','Attendance Day 5']].mean(axis=1) 

In [ ]:
attendance_per

In [ ]:

student_performance['Weighted Percentage'] = round(attendance_per * 0.4 +
                                       student_performance['percentage_mini_test1'] * 0.1 +
                                       student_performance['percentage_mini_test2'] * 0.1 +
                                       student_performance['percentage_live_test'] * 0.2 +
                                       student_performance['Assignment'] * 0.2)*100

student_performance['Weighted Percentage']

In [ ]:
student_performance['Weighted Percentage'].unique

In [ ]:
def performance_category(df):
    if df['Percentage']>=85:
        return "Excellent" 
    elif df['Percentage']>=71 and df['Weighted Percentage']<=84:
        return "Good"
    elif df['Weighted Percentage']>=50 and df['Weighted Percentage']<=70:
        return "Average"
    else:
        return "Needs Improvement" 

In [ ]:
student_performance['Performance'] = student_performance.apply(performance_category,axis=1)

In [ ]:
##student_performance.to_csv("C:/Data_Science/Pandas/Marks_analysis_v2.csv")

In [ ]:
student_performance

In [ ]:
##1.Identify students with attendance below 75% but weighted percentage >50%.
student_performance[(student_performance['Attendance Percentage']<75) & (student_performance['Weighted Percentage']>50)]

In [ ]:
#2.Highlight the top three students based on percentage marks.
top3_student = student_performance.nlargest(3,columns='Percentage')
print(top3_student[['Name','Percentage','Total Marks','Performance','Attendance Percentage']])

In [ ]:
#3.	Impact of attendance on Tests/Assignment marks. 
student_performance[['Mini Test 1','Mini Test 2','Live Test','Assignment','Total Attendance']].corr()

In [ ]:
#1.	Create a bar chart displaying weighted percentages for top 5 students.

top5_student = student_performance.nlargest(5,columns='Weighted Percentage')
print(top5_student[['Name','Weighted Percentage']])

In [ ]:
custom_colors = ['#34495e', '#3498db', '#b7950b', '#fdebd0', '#FFA500'] 
fig = px.bar(top5_student, x="Name", y="Weighted Percentage", barmode="group",title='weighted percentages for top 5 students',color_discrete_sequence=custom_colors)
fig.update_layout(
    title={
        'text': 'weighted % for top 5 students',
        'y': 0.89,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font_size=13,
    height=400,
    width=500,
)
fig.show()

In [ ]:
#2.	Create a pie chart showing the distribution of students across the four performance categories.

In [ ]:
performance_df = pd.DataFrame(student_performance['Performance'].value_counts()).reset_index()
performance_df.columns=['Performance','Count']

In [ ]:
performance_df

In [ ]:
# pie chart
custom_colors = ['#34495e', '#3498db', '#b7950b', '#fdebd0']
fig = px.pie(performance_df, values='Count', names='Performance', title='Distribution of Students Across Performance Categories',color_discrete_sequence=custom_colors)
fig.update_layout(
    title={
        'text': 'Distribution of Students Across Performance Categories',
        'y': 0.98,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font_size=15,
    paper_bgcolor='lightgrey', 
    height=400,
    width=500,
)
fig.show()

In [ ]:
#3.	Create box plots for each test (Live Test, Mini Test 1, Mini Test 2, Assignment) to visualize the spread and detect potential outliers in scores.

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('Box Plots for Each Test', fontsize=20)

sns.boxplot(y=student_performance['Mini Test 1'], ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Mini Test 1')
axes[0, 0].set_ylabel('Marks')

sns.boxplot(y=student_performance['Mini Test 2'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Mini Test 2')
axes[0, 1].set_ylabel('Marks')

sns.boxplot(y=student_performance['Live Test'], ax=axes[1, 0], color='lightcoral')
axes[1, 0].set_title('Live Test')
axes[1, 0].set_ylabel('Marks')

sns.boxplot(y=student_performance['Assignment'], ax=axes[1, 1], color='pink')
axes[1, 1].set_title('Assignment')
axes[1, 1].set_ylabel('Marks')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
#4.	Create a chart to show the students where attendance is less than 50%.

attendance_less_than_50 = student_performance[student_performance['Attendance Percentage']<50]
print(attendance_less_than_50[['Name','Attendance Percentage']])

In [ ]:
fig = px.bar(attendance_less_than_50, x="Name", y="Attendance Percentage", barmode="group",title='Attendance less than 50%')
fig.update_layout(
    title={
        'y': 0.88,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font_size=13,
    paper_bgcolor='lightyellow', 
    height=600,
    width=1120,
)
fig.show()